## Lab 5: Recap on Dimensionality Reduction Techniques

In this lab we apply all the dimensionality reduction techniques seen so far (PCA, Kernel PCA, Isomap) plus t-SNE, UMAP and the autoencoder to the same dataset, and we compare them.

You are required to use **your own implementations** of PCA, Kernel PCA and Isomap from the previous labs, and to write **your own autoencoder** with the layers of `pytorch`. For t-SNE and UMAP you can use external libraries.

The first part of this notebook (download and preprocessing of the dataset) is already written for you: run it, do not modify it, and write your solution below.

In [ ]:
import os
import gzip
import struct
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

np.random.seed(0)

# Exercise 1
### Download the dataset
The `Fashion-MNIST` dataset ([link](https://github.com/zalandoresearch/fashion-mnist#get-the-data)) is made of $28\times28$ grayscale images of clothes, belonging to 10 classes. The four files are downloaded in `../Datasets/fashion_mnist` (only the first time you run the notebook).

In [ ]:
DATA_DIR = "../Datasets/fashion_mnist"
BASE_URL = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
FILES = ["train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz",
         "t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"]

os.makedirs(DATA_DIR, exist_ok=True)
for f in FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        print("downloading", f)
        urllib.request.urlretrieve(BASE_URL + f, path)   # if this fails, download the 4 files by hand from the link above
print("files in", DATA_DIR, ":", sorted(os.listdir(DATA_DIR)))

In [ ]:
def read_idx(filename):
    """Read a file in the IDX format (used by MNIST-like datasets)."""
    with gzip.open(filename, 'rb') as f:
        zero, data_type, dims = struct.unpack('>HBB', f.read(4))
        shape = tuple(struct.unpack('>I', f.read(4))[0] for d in range(dims))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(shape)

Xtrain_full = read_idx(os.path.join(DATA_DIR, 'train-images-idx3-ubyte.gz'))
ytrain_full = read_idx(os.path.join(DATA_DIR, 'train-labels-idx1-ubyte.gz'))
Xtest_full  = read_idx(os.path.join(DATA_DIR, 't10k-images-idx3-ubyte.gz'))
ytest_full  = read_idx(os.path.join(DATA_DIR, 't10k-labels-idx1-ubyte.gz'))

print(Xtrain_full.shape, ytrain_full.shape, Xtest_full.shape, ytest_full.shape)

In [ ]:
labels_dict = {0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat',
               5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'}

### Preprocessing
1. **Undersampling**: 5000 images for training and 1000 for testing (random, stratified, so that all classes are equally represented). Some algorithms (kernel PCA, Isomap, t-SNE) need the $n\times n$ matrix of the pairwise relations, and they would be too slow on 60000 points.
2. **Rescaling** of the pixels to $[0,1]$.
3. **Centering**: we subtract the mean image of the *training set* from both the training and the test set (the test set must not be used to compute any statistic).
4. **Flattening**: every image becomes a vector of $784$ pixels (the format needed by all the algorithms of today).

In [ ]:
Xtrain_img, _, ytrain, _ = train_test_split(Xtrain_full, ytrain_full, train_size=5000, stratify=ytrain_full, random_state=0)
_, Xtest_img, _, ytest   = train_test_split(Xtest_full, ytest_full, test_size=1000, stratify=ytest_full, random_state=0)

# rescale to [0, 1]
Xtrain_img = Xtrain_img / 255
Xtest_img = Xtest_img / 255

# center with the mean of the TRAIN set
mean_img = Xtrain_img.mean(axis=0)            # (28, 28), useful to visualize the images after the centering
Xtrain = Xtrain_img - mean_img
Xtest = Xtest_img - mean_img

# flatten: (n, 28, 28) -> (n, 784)
Xtrain = Xtrain.reshape(Xtrain.shape[0], -1)
Xtest = Xtest.reshape(Xtest.shape[0], -1)

print("Xtrain:", Xtrain.shape, " Xtest:", Xtest.shape)
print("mean of the train pixels (should be ~0):", Xtrain.mean().round(6))
print("class counts in train:", np.bincount(ytrain))

The variables you will work with are `Xtrain` ($5000\times784$) and `Xtest` ($1000\times784$), with the labels `ytrain` and `ytest` (use the labels **only** for the colours of the plots and for the evaluation of the results, never to compute the embeddings). The dictionary `labels_dict` gives the name of each class, and `mean_img` can be added back to a reshaped image (`x.reshape(28, 28) + mean_img`) to display it in its original colors.

Let us have a look at the data.

In [ ]:
n_row, n_col = 4, 5
fig, ax = plt.subplots(n_row, n_col, figsize=(1.5*n_col, 2*n_row))
for i in range(n_row * n_col):
    ax_ = ax[i // n_col, i % n_col]
    ax_.imshow(Xtrain_img[i], cmap='gray')
    ax_.set_title(labels_dict[ytrain[i]], fontsize=9)
    ax_.axis('off')
plt.tight_layout()
plt.show()

# Your solution
Apply the techniques below to `Xtrain` (see the text of the lab for the details). Reuse your implementations of the previous labs, and remember the goal: find the best *hyperparameter-method* combination and discuss the differences.

### PCA (own implementation)

### Kernel PCA (own implementation): Gaussian and polynomial kernel

### Isomap (own implementation)

### t-SNE

### UMAP

### Autoencoder (`pytorch` layers)

### Comparison and discussion